# Actividad Extra: Cartas Magic: The Gathering

## Objetivos
- Leer y explorar un dataset de cartas de Magic: The Gathering
- Analizar la distribucion por tipo y rareza de cartas
- Explorar el coste de mana convertido (CMC)
- Aplicar filtros avanzados para encontrar cartas especificas
- Realizar agrupaciones por set/expansion
- Crear visualizaciones relevantes

## Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import pandas as pd

spark = SparkSession.builder \
    .appName("actividad_cartas_magic") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

---
## Ejercicio 1: Lectura y exploracion del dataset

Lee el archivo `cartas_magic.csv`, explora el schema y obtiene estadisticas descriptivas.

**Columnas del dataset:** cardName, cardCmc, cardType, creatureType, powTough, set, setNum, rarity, language, cardImage

In [ ]:
# =============================================================
# EJERCICIO 1: Lectura y exploracion
# =============================================================

# Leer CSV
df = spark.read.csv("/home/jovyan/datos/cartas_magic.csv", header=True, inferSchema=True)

# Schema
df.printSchema()

# Primeras filas
df.show(10, truncate=50)

# Total de registros
print(f"Total de cartas: {df.count()}")

# Convertir cardCmc a numerico si es string
df = df.withColumn("cardCmc", F.col("cardCmc").cast("double"))

# Estadisticas descriptivas
print("\nEstadisticas descriptivas:")
df.describe().show(truncate=False)

> **Nota docente:** El dataset de Magic contiene informacion sobre cartas del juego de cartas coleccionables. La columna `cardCmc` (Converted Mana Cost) puede llegar como string si tiene valores mixtos, por lo que es necesario convertirla a double. `inferSchema=True` hace su mejor esfuerzo pero no siempre acierta. `powTough` contiene poder/resistencia en formato "X/Y" para criaturas y puede ser null para otros tipos de cartas. `set` es una palabra reservada en Python pero no causa problemas como nombre de columna en Spark (se accede con `F.col("set")`).

---
## Ejercicio 2: Distribucion por tipo y rareza

Analiza la distribucion de cartas por `cardType` y por `rarity`. Crea graficos de barras para cada uno.

In [ ]:
# =============================================================
# EJERCICIO 2: Distribucion por tipo y rareza
# =============================================================

# --- Distribucion por tipo de carta ---
df_tipos = df.groupBy("cardType").count() \
    .orderBy("count", ascending=False) \
    .limit(10)

print("Top 10 tipos de carta:")
df_tipos.show(truncate=False)

pdf_tipos = df_tipos.toPandas()

plt.figure(figsize=(12, 5))
plt.barh(pdf_tipos["cardType"].fillna("Desconocido"), pdf_tipos["count"], color="#8B4513")
plt.title("Top 10 Tipos de Carta", fontsize=14)
plt.xlabel("Cantidad")
plt.ylabel("Tipo")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# --- Distribucion por rareza ---
df_rareza = df.groupBy("rarity").count() \
    .orderBy("count", ascending=False)

print("Distribucion por rareza:")
df_rareza.show(truncate=False)

pdf_rareza = df_rareza.toPandas()

colores_rareza = ["#A0A0A0", "#C0C0C0", "#FFD700", "#FF6600", "#800080"]

plt.figure(figsize=(8, 5))
plt.bar(pdf_rareza["rarity"].fillna("N/A"), pdf_rareza["count"],
        color=colores_rareza[:len(pdf_rareza)])
plt.title("Distribucion por Rareza", fontsize=14)
plt.xlabel("Rareza")
plt.ylabel("Cantidad")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

> **Nota docente:** Los tipos de carta en Magic incluyen Creature, Instant, Sorcery, Enchantment, Artifact, Land, Planeswalker, etc. Las criaturas son el tipo mas comun. Las rarezas tipicas son Common (comun), Uncommon (poco comun), Rare (rara) y Mythic Rare (mitica). Los colores del grafico de rareza intentan representar los colores asociados al simbolo de expansion: gris para Common, plateado para Uncommon, dorado para Rare, naranja para Mythic. Los alumnos pueden investigar por que la distribucion de rareza sigue este patron (diseno de juego y economia de sobres).

---
## Ejercicio 3: Analisis del coste de mana convertido (CMC)

Analiza la columna `cardCmc` (Converted Mana Cost): histograma de distribucion, promedio por tipo de carta y promedio por rareza.

In [ ]:
# =============================================================
# EJERCICIO 3: Analisis del CMC
# =============================================================

# Filtrar nulls de CMC
df_cmc = df.filter(F.col("cardCmc").isNotNull())

# Histograma de CMC
pdf_cmc = df_cmc.select("cardCmc").toPandas()

plt.figure(figsize=(10, 5))
plt.hist(pdf_cmc["cardCmc"].dropna(), bins=20, color="#8B4513", edgecolor="white", alpha=0.8)
plt.title("Distribucion del Coste de Mana Convertido (CMC)", fontsize=14)
plt.xlabel("CMC")
plt.ylabel("Frecuencia")
plt.axvline(pdf_cmc["cardCmc"].mean(), color="red", linestyle="--",
            label=f"Media: {pdf_cmc['cardCmc'].mean():.1f}")
plt.legend()
plt.tight_layout()
plt.show()

# CMC promedio por tipo de carta (top 10)
print("CMC promedio por tipo de carta:")
df_cmc.groupBy("cardType").agg(
    F.avg("cardCmc").alias("cmc_promedio"),
    F.count("*").alias("num_cartas")
).filter(F.col("num_cartas") >= 10) \
 .orderBy("cmc_promedio", ascending=False) \
 .show(10, truncate=False)

# CMC promedio por rareza
print("CMC promedio por rareza:")
df_cmc.groupBy("rarity").agg(
    F.avg("cardCmc").alias("cmc_promedio"),
    F.count("*").alias("num_cartas")
).orderBy("cmc_promedio", ascending=False).show()

> **Nota docente:** El CMC (Converted Mana Cost, ahora llamado Mana Value) es una metrica central en Magic que indica el coste total de una carta. La distribucion suele concentrarse en valores bajos (1-4) con una cola hacia la derecha (cartas caras como criaturas legendarias). El filtro `num_cartas >= 10` evita que tipos con pocas cartas distorsionen el analisis. Las cartas de mayor rareza (Mythic Rare) tienden a tener un CMC ligeramente mayor, lo cual refleja que las cartas poderosas y costosas suelen ser mas raras.

---
## Ejercicio 4: Filtros avanzados

Encuentra cartas con caracteristicas especificas:
- Cartas con CMC >= 7 (cartas de alto coste)
- Cartas de tipo "Creature" con rareza "Mythic Rare"
- Cartas cuyo nombre contenga una palabra especifica

In [ ]:
# =============================================================
# EJERCICIO 4: Filtros avanzados
# =============================================================

# Cartas de alto coste (CMC >= 7)
print("=== Cartas de alto coste (CMC >= 7) ===")
df_alto_coste = df.filter(F.col("cardCmc") >= 7)
print(f"Total: {df_alto_coste.count()} cartas")
df_alto_coste.select("cardName", "cardType", "cardCmc", "rarity").show(10, truncate=False)

# Criaturas miticas
print("=== Criaturas con rareza Mythic Rare ===")
df_miticas = df.filter(
    (F.col("cardType").contains("Creature")) &
    (F.col("rarity") == "mythic")
)
print(f"Total: {df_miticas.count()} cartas")
df_miticas.select("cardName", "cardType", "creatureType", "cardCmc", "powTough").show(10, truncate=False)

# Buscar cartas con "Dragon" en el nombre
print("=== Cartas con 'Dragon' en el nombre ===")
df_dragones = df.filter(F.col("cardName").contains("Dragon"))
print(f"Total: {df_dragones.count()} cartas")
df_dragones.select("cardName", "cardType", "cardCmc", "rarity", "set").show(10, truncate=False)

> **Nota docente:** Los filtros avanzados combinan multiples condiciones con `&` (AND) y `|` (OR). `contains()` es case-sensitive, por lo que "Dragon" no encontrara "dragon". Para busqueda case-insensitive se puede usar `F.lower(F.col("cardName")).contains("dragon")`. El valor de rarity puede ser "mythic" en minuscula en este dataset, por lo que es importante verificar los valores reales antes de filtrar. Los alumnos deben recordar que cada condicion debe ir entre parentesis cuando se combinan con `&` o `|` debido a la precedencia de operadores en Python.

---
## Ejercicio 5: Agrupaciones por set/expansion

Analiza como se distribuyen las cartas por `set` (expansion). Encuentra los sets con mas cartas y el CMC promedio por set.

In [ ]:
# =============================================================
# EJERCICIO 5: Agrupaciones por set
# =============================================================

# Sets con mas cartas y CMC promedio
df_sets = df.groupBy("set").agg(
    F.count("*").alias("num_cartas"),
    F.avg("cardCmc").alias("cmc_promedio"),
    F.countDistinct("rarity").alias("num_rarezas")
).orderBy("num_cartas", ascending=False)

print("Top 15 sets por numero de cartas:")
df_sets.show(15, truncate=False)

# Grafico de barras
pdf_sets = df_sets.limit(15).toPandas()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Barras: numero de cartas por set
ax1.barh(pdf_sets["set"].fillna("N/A"), pdf_sets["num_cartas"], color="#8B4513")
ax1.set_title("Top 15 Sets por Numero de Cartas", fontsize=12)
ax1.set_xlabel("Numero de Cartas")
ax1.invert_yaxis()

# Barras: CMC promedio por set
ax2.barh(pdf_sets["set"].fillna("N/A"), pdf_sets["cmc_promedio"].fillna(0), color="#DAA520")
ax2.set_title("CMC Promedio por Set (Top 15)", fontsize=12)
ax2.set_xlabel("CMC Promedio")
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

> **Nota docente:** `F.countDistinct()` cuenta valores unicos, util para saber cuantas rarezas diferentes tiene cada set. El uso de `subplots` permite mostrar dos graficos lado a lado para comparar visualmente el tamano del set con su CMC promedio. Los sets mas grandes suelen ser los sets base (Core Sets) o sets de expansion principales. `fillna()` en pandas es necesario para evitar errores de matplotlib con valores NaN. Los alumnos avanzados pueden investigar la relacion entre el tamano del set y la complejidad promedio de las cartas.

---
## Ejercicio 6: Visualizaciones

Crea 2-3 visualizaciones relevantes:
1. Distribucion de rareza por tipo de carta (barras apiladas o agrupadas)
2. Top 10 tipos de criaturas mas comunes
3. Boxplot del CMC por rareza

In [ ]:
# =============================================================
# EJERCICIO 6: Visualizaciones
# =============================================================

# --- Visualizacion 1: Distribucion de rareza (pastel) ---
pdf_rareza = df.groupBy("rarity").count().orderBy("count", ascending=False).toPandas()

plt.figure(figsize=(8, 8))
colores = ["#A0A0A0", "#C0C0C0", "#FFD700", "#FF6600", "#800080", "#4169E1"]
plt.pie(pdf_rareza["count"], labels=pdf_rareza["rarity"].fillna("N/A"),
        autopct="%1.1f%%", colors=colores[:len(pdf_rareza)], startangle=90)
plt.title("Distribucion de Cartas por Rareza", fontsize=14)
plt.tight_layout()
plt.show()

# --- Visualizacion 2: Top 10 tipos de criaturas ---
df_criaturas = df.filter(F.col("creatureType").isNotNull()) \
    .filter(F.col("creatureType") != "") \
    .groupBy("creatureType").count() \
    .orderBy("count", ascending=False) \
    .limit(10)

pdf_criaturas = df_criaturas.toPandas()

plt.figure(figsize=(10, 6))
plt.barh(pdf_criaturas["creatureType"], pdf_criaturas["count"], color="#2E7D32")
plt.title("Top 10 Tipos de Criaturas", fontsize=14)
plt.xlabel("Cantidad")
plt.ylabel("Tipo de Criatura")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# --- Visualizacion 3: Boxplot CMC por rareza ---
pdf_box = df.filter(F.col("cardCmc").isNotNull()) \
    .filter(F.col("rarity").isNotNull()) \
    .select("rarity", "cardCmc").toPandas()

rarezas_orden = pdf_box["rarity"].value_counts().index.tolist()
data_box = [pdf_box[pdf_box["rarity"] == r]["cardCmc"].dropna().values for r in rarezas_orden]

plt.figure(figsize=(10, 6))
bp = plt.boxplot(data_box, labels=rarezas_orden, patch_artist=True)
colores_box = ["#A0A0A0", "#C0C0C0", "#FFD700", "#FF6600", "#800080", "#4169E1"]
for patch, color in zip(bp["boxes"], colores_box[:len(bp["boxes"])]):
    patch.set_facecolor(color)
plt.title("Distribucion del CMC por Rareza", fontsize=14)
plt.xlabel("Rareza")
plt.ylabel("CMC")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

> **Nota docente:** Las tres visualizaciones muestran diferentes aspectos del dataset. El grafico de pastel es adecuado cuando hay pocas categorias (4-6 rarezas). Para muchas categorias seria mejor usar barras. El boxplot es excelente para comparar distribuciones entre grupos, mostrando mediana, cuartiles y outliers. Los tipos de criaturas mas comunes en Magic suelen ser Human, Soldier, Wizard, etc. Los alumnos pueden experimentar con otros tipos de graficos de matplotlib como violin plots (`plt.violinplot`) para una representacion mas detallada de la distribucion.

---
## Resumen

En esta actividad aprendimos:

1. **Exploracion de datos** con schema, describe y conteos
2. **Distribucion de categorias** con groupBy y graficos
3. **Analisis numerico** del coste de mana convertido
4. **Filtros avanzados** con multiples condiciones y contains()
5. **Agrupaciones** por set/expansion con metricas multiples
6. **Visualizaciones variadas** incluyendo pastel, barras y boxplot

In [ ]:
# Detener SparkSession
spark.stop()
print("SparkSession detenida correctamente.")